# EX: Simulating Value Iteration

In this exercise, we will program a simplified Value Iteration loop. An AI agent is operating in a 3-sector tactical grid (`Base`, `Contested`, `Target`). 

We will automate the Bellman update to observe how the state values physically converge over multiple iterations as the agent mathematically looks further into the future.

**Lab Steps:**
*   **Initialize** the values of all states to 0.
*   **Define the Bellman update loop**, factoring in transition probabilities, immediate rewards, and the discount factor.
*   **Observe the convergence** of the state values over successive iterations.

In [1]:
# Define the tactical environment variables
states = ['Base', 'Contested', 'Target']
V = {'Base': 0.0, 'Contested': 0.0, 'Target': 0.0}  # V_0(s) = 0
gamma = 0.9
iterations = 50
convergence_threshold = 0.01

print(f"Iteration 0: Base={V['Base']:.3f}, Contested={V['Contested']:.3f}, Target={V['Target']:.3f}")

for i in range(1, iterations + 1):
    V_new = V.copy()
    
    # --- Bellman Update for 'Contested' Sector ---
    # Action: Push to Target (100% success chance)
    # Reward = +10. Target is a terminal state (value remains 0).
    q_push = 1.0 * (10 + gamma * V['Target'])
    
    # Action: Retreat to Base (100% success chance)
    # Reward = 0.
    q_retreat = 1.0 * (0 + gamma * V['Base'])
    
    V_new['Contested'] = max(q_push, q_retreat)
    
    # --- Bellman Update for 'Base' Sector ---
    # Action: Move to Contested (100% success chance)
    # Reward = 0.
    q_move = 1.0 * (0 + gamma * V['Contested'])
    V_new['Base'] = q_move
    
    # Check for mathematical convergence
    max_change = max(abs(V_new[s] - V[s]) for s in states)
    V = V_new
    
    if i <= 4:
        print(f"Iteration {i}: Base={V['Base']:.3f}, Contested={V['Contested']:.3f}, Target={V['Target']:.3f}")
        
    if max_change < convergence_threshold:
        print("...")
        print(f"Values Converged at Iteration {i}!")
        print(f"Final Values: Base={V['Base']:.3f}, Contested={V['Contested']:.3f}, Target={V['Target']:.3f}")
        break

Iteration 0: Base=0.000, Contested=0.000, Target=0.000
Iteration 1: Base=0.000, Contested=10.000, Target=0.000
Iteration 2: Base=9.000, Contested=10.000, Target=0.000
Iteration 3: Base=9.000, Contested=18.100, Target=0.000
Iteration 4: Base=16.290, Contested=18.100, Target=0.000
...
Values Converged at Iteration 42!
Final Values: Base=89.913, Contested=99.904, Target=0.000


## Interpreting the Results

At `Iteration 0`, the agent has a "zero-step horizon." It cannot see any future rewards, so all values are mathematically zero.

At `Iteration 1`, the agent performs a one-step lookahead. The `Contested` sector detects the immediate +10 reward of pushing to the `Target`.

At `Iteration 2`, the agent performs a two-step lookahead. The `Base` sector finally "sees" the value of the `Contested` sector through the transition mathematics, updating its own state value to 9.0 (10 discounted by $\gamma = 0.9$).

This ripple effect mathematically continues until the changes become infinitesimally small, successfully converging at Iteration 42. The AI now has the optimal $V^*(s)$ values required to execute Policy Extraction.